# 02 - Cleaning & Data Integrity

"Already split" and "from Kaggle" guarantee neither quality nor validity. EDA already hit a **truncated image** (crashed PIL), **100-megapixel** images, and **non-RGB** modes. Here we verify integrity and protect downstream results from two failure modes:

1. **Corrupt / truncated / unreadable** images → detected with a *strict* decode and dropped.
2. **Duplicates & train↔test leakage** → if the same (or near-identical) image is in both train and test, test scores are inflated and meaningless. We detect **exact** duplicates (content hash) and **near** duplicates (perceptual hash), and remove cross-split overlap.

Output per dataset: **`data/<name>/manifest_clean.csv`** with a `keep` column (we never delete source files), plus a short report.

**Sections:** 0 Setup · 1 Scan images · 2 Corrupt / oversized · 3 Duplicates & leakage · 4 Cleaned manifest.

> The scan reads every image once. `ai-real-images` is ~48 GB, so a full run takes a while (JPEG decoding is sped up via `draft`; PNGs in `tiny-genimage` decode fully). Set `LIMIT` to an int for a quick dry-run.

## 0 - Setup

Load both manifests and configure the scan (worker threads for the I/O-bound read, and an optional row limit for a dry-run).

In [1]:
import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

_here = Path.cwd()
_nb_dir = _here if (_here / "utils").is_dir() else _here / "notebooks"
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

from utils import data as dutils
from utils import clean
from utils.paths import repo_paths

PATHS = repo_paths(_nb_dir)
DATA_DIR = PATHS["data"]

LIMIT = None      # set to e.g. 2000 for a quick dry-run (first N rows per dataset)
N_WORKERS = 8     # threads for the I/O-bound scan
GIANT_PIXELS = 50_000_000  # flag images larger than this many pixels

DATASETS = list(dutils.KAGGLE_SLUGS)
manifests = {name: pd.read_csv(DATA_DIR / name / "manifest.csv") for name in DATASETS}
for name, m in manifests.items():
    print(f"{name:16s} {len(m):>7,} rows")

ai-real-images    60,000 rows
tiny-genimage     35,000 rows


## 1 - Scan images

One pass per dataset: each image is read once and `clean.scan_image` returns whether it decodes (strictly — truncated files fail), its content hash (`sha1`), its perceptual hash (`phash`), and its true dimensions/mode. We join these columns onto the manifest.

In [2]:
def scan_manifest(m):
    paths = m["filepath"].tolist()
    if LIMIT:
        paths = paths[:LIMIT]
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        recs = list(tqdm(ex.map(clean.scan_image, paths), total=len(paths)))
    scan = pd.DataFrame(recs)
    return m.iloc[:len(paths)].reset_index(drop=True).join(scan)

scans = {}
for name, m in manifests.items():
    print(f"scanning {name} ...")
    scans[name] = scan_manifest(m)
print("done.")

scanning ai-real-images ...


  0%|          | 0/60000 [00:00<?, ?it/s]

C:\Users\ΜΑΤΟΥΛΑ\Documents\Deepfake Detection\.venv\Lib\site-packages\PIL\Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


scanning tiny-genimage ...


  0%|          | 0/35000 [00:00<?, ?it/s]

done.


## 2 - Corrupt / oversized

List unreadable files (these get dropped) and flag oversized / non-RGB images (informational — the loaders convert to RGB, but huge images are worth knowing about).

In [3]:
for name, s in scans.items():
    bad = s[~s["readable"]]
    giant = s[s["readable"] & ((s["width"] * s["height"]) > GIANT_PIXELS)]
    nonrgb = s[s["readable"] & (s["mode"] != "RGB")]
    print(f"==== {name} ====")
    print(f"  unreadable : {len(bad)}")
    print(f"  oversized  : {len(giant)}  (> {GIANT_PIXELS:,} px)")
    print(f"  non-RGB    : {len(nonrgb)}  modes={nonrgb['mode'].value_counts().to_dict()}")
    if len(bad):
        display(bad[["dataset", "split", "label", "filename", "error"]].head(20))

==== ai-real-images ====
  unreadable : 12
  oversized  : 189  (> 50,000,000 px)
  non-RGB    : 9263  modes={'P': 9054, 'RGBA': 163, 'L': 42, 'CMYK': 4}


,dataset,split,label,filename,error
11196,ai-real-images,test,real,5197.jpg,OSError: image file is truncated (6 bytes not ...
11324,ai-real-images,test,real,5325.jpg,OSError: image file is truncated (3 bytes not ...
11878,ai-real-images,test,real,5879.jpg,OSError: image file is truncated
16139,ai-real-images,train,fake,12854.jpg,OSError: broken data stream when reading image...
34022,ai-real-images,train,fake,8022.jpg,OSError: broken data stream when reading image...
36037,ai-real-images,train,real,0038.jpg,OSError: image file is truncated (0 bytes not ...
39303,ai-real-images,train,real,12094.jpg,OSError: image file is truncated (6 bytes not ...
40323,ai-real-images,train,real,13021.jpg,OSError: image file is truncated (0 bytes not ...
43559,ai-real-images,train,real,15963.jpg,OSError: image file is truncated (2 bytes not ...
43612,ai-real-images,train,real,16011.jpg,OSError: image file is truncated (4 bytes not ...


==== tiny-genimage ====
  unreadable : 0
  oversized  : 0  (> 50,000,000 px)
  non-RGB    : 2780  modes={'RGBA': 2500, 'L': 280}


## 3 - Duplicates & train↔test leakage

Among readable images we group by **content hash** (exact duplicates) and by **perceptual hash** (near-duplicates — resaves / resizes that look identical). The dangerous case is a hash that appears in **more than one split**: that's leakage. We report counts and show example leaking groups.

In [4]:
def dup_report(s):
    ok = s[s["readable"]]
    exact = ok.groupby("sha1").size()
    near = ok.groupby("phash").size()
    leak_exact = ok.groupby("sha1")["split"].nunique()
    leak_near = ok.groupby("phash")["split"].nunique()
    return {
        "exact_dup_rows": int((exact[exact > 1]).sum() - (exact > 1).sum()),  # rows beyond the first
        "near_dup_rows": int((near[near > 1]).sum() - (near > 1).sum()),
        "cross_split_exact_groups": int((leak_exact > 1).sum()),
        "cross_split_near_groups": int((leak_near > 1).sum()),
        "_leak_hashes": leak_near[leak_near > 1].index.tolist(),
    }

reports = {}
for name, s in scans.items():
    r = dup_report(s)
    reports[name] = r
    print(f"==== {name} ====")
    print(f"  exact-duplicate rows (beyond first) : {r['exact_dup_rows']}")
    print(f"  near-duplicate rows  (beyond first) : {r['near_dup_rows']}")
    print(f"  cross-split EXACT leakage groups    : {r['cross_split_exact_groups']}")
    print(f"  cross-split NEAR  leakage groups    : {r['cross_split_near_groups']}")

==== ai-real-images ====
  exact-duplicate rows (beyond first) : 41
  near-duplicate rows  (beyond first) : 106
  cross-split EXACT leakage groups    : 12
  cross-split NEAR  leakage groups    : 27
==== tiny-genimage ====
  exact-duplicate rows (beyond first) : 1
  near-duplicate rows  (beyond first) : 2
  cross-split EXACT leakage groups    : 0
  cross-split NEAR  leakage groups    : 1


In [5]:
# Inspect an example near-duplicate leakage group, if any (same pHash across splits).
name = "ai-real-images"
leak_hashes = reports[name]["_leak_hashes"]
if leak_hashes:
    g = scans[name][scans[name]["phash"] == leak_hashes[0]]
    display(g[["split", "label", "source", "filename", "sha1", "phash"]])
else:
    print(f"No cross-split leakage in {name} — good.")

,split,label,source,filename,sha1,phash
6263,test,real,NaN,0264.jpg,dfaa380a3129e70a88fe53d557480b7cf3cabfd7,84503b254efa35d7
40102,train,real,NaN,12820.jpg,dfaa380a3129e70a88fe53d557480b7cf3cabfd7,84503b254efa35d7


## 4 - Cleaned manifest

Build the `keep` flag with clear rules:

1. **Drop unreadable** images.
2. **Keep one** representative per exact-duplicate (`sha1`) and per near-duplicate (`phash`) group.
3. For cross-split duplicates, prefer keeping the **train** copy (split priority `train < val < test`) so the **test/val set never contains a memorized training image**.

We sort by split priority, keep the first per hash, then restore original order and save `manifest_clean.csv`.

In [6]:
SPLIT_PRIORITY = {"train": 0, "val": 1, "test": 2}

def build_clean(s):
    s = s.copy()
    s["_order"] = s["split"].map(SPLIT_PRIORITY).fillna(3).astype(int)
    # Process readable rows, train-first, so the kept copy of any duplicate is the train one.
    ok = s[s["readable"]].sort_values(["_order"]).copy()
    keep_idx = set()
    seen_sha, seen_phash = set(), set()
    for idx, sha, ph in zip(ok.index, ok["sha1"], ok["phash"]):
        if sha in seen_sha or ph in seen_phash:
            continue  # duplicate of something we already kept
        seen_sha.add(sha); seen_phash.add(ph)
        keep_idx.add(idx)
    s["keep"] = s.index.isin(keep_idx)
    return s.drop(columns="_order")

cleaned = {name: build_clean(s) for name, s in scans.items()}
for name, s in cleaned.items():
    dropped = (~s["keep"]).sum()
    print(f"{name:16s} keep {int(s['keep'].sum()):>7,} / {len(s):>7,}  (dropped {int(dropped):,})")

ai-real-images   keep  59,882 /  60,000  (dropped 118)
tiny-genimage    keep  34,998 /  35,000  (dropped 2)


In [7]:
# Per-dataset breakdown of why rows were dropped, then save the cleaned manifest.
for name, s in cleaned.items():
    n_corrupt = int((~s["readable"]).sum())
    n_dropped = int((~s["keep"]).sum())
    n_dup = n_dropped - n_corrupt
    print(f"{name:16s} dropped {n_dropped:,}  (corrupt {n_corrupt:,} + duplicate/leak {n_dup:,})")
    # Sanity: confirm no kept hash spans multiple splits.
    kept = s[s["keep"]]
    leak_left = (kept.groupby("phash")["split"].nunique() > 1).sum()
    print(f"                 remaining cross-split near-dup groups: {int(leak_left)} (should be 0)")
    out = DATA_DIR / name / "manifest_clean.csv"
    s.to_csv(out, index=False)
    print(f"                 saved -> {out}")

ai-real-images   dropped 118  (corrupt 12 + duplicate/leak 106)
                 remaining cross-split near-dup groups: 0 (should be 0)
                 saved -> C:\Users\ΜΑΤΟΥΛΑ\Documents\Deepfake Detection\notebooks\data\ai-real-images\manifest_clean.csv
tiny-genimage    dropped 2  (corrupt 0 + duplicate/leak 2)
                 remaining cross-split near-dup groups: 0 (should be 0)
                 saved -> C:\Users\ΜΑΤΟΥΛΑ\Documents\Deepfake Detection\notebooks\data\tiny-genimage\manifest_clean.csv


**Result:** every later notebook should read `manifest_clean.csv` and filter to `keep == True`. This guarantees no corrupt files crash training and no train/test leakage inflates results.

**Next:** `03_split_and_preprocessing.ipynb` — carve a stratified validation split from `ai-real-images` train, compute training-set normalization stats, and define the canonical resize/transforms + `Dataset`/`DataLoader`.